# Лабораторная работа №5. Градиентный бустинг

Чибугаев Илья Алексеевич, вариант 22.

## Данные и метрики

Для классификации взят [Haberman's Survival](https://archive.ics.uci.edu/dataset/43/haberman+s+survival): 306 пациентов и 3 признака. Целевой класс `1` означает смерть в течение пяти лет после операции. По возрасту, году операции и числу пораженных лимфоузлов нужно оценить этот риск.

Для регрессии взят [QSAR aquatic toxicity](https://archive.ics.uci.edu/dataset/505/qsar+aquatic+toxicity): 546 химических соединений и 8 молекулярных дескрипторов. Целевая переменная: токсичность LC50 для Daphnia magna. Такой прогноз помогает отсеивать опасные вещества до экспериментов.

Бустинг добавляет деревья последовательно. Каждое следующее исправляет ошибку текущего ансамбля: остаток в регрессии или градиент логистической потери в классификации.

Train и test соотносятся как 80/20, `random_state=22`. Классы при разбиении сохраняются в одинаковой пропорции. Подбор параметров идет только на train.

В классификации основной показатель: ROC-AUC. Он не зависит от выбранного порога. F1 считается для класса `1`, accuracy показывает общую долю верных ответов. В регрессии основной показатель: RMSE в единицах LC50. MAE показывает типичную абсолютную ошибку, R² показывает долю объясненной дисперсии.


## Подготовка

Общий код вынесен в `src`, чтобы не повторять его в каждой работе.

In [1]:
# Подключаю общие функции и настраиваю вывод таблиц.
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.experiments import TaskExperiment

pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)
plt.style.use("seaborn-v0_8-whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)
print(f"Корень проекта: {ROOT}")

Корень проекта: /Users/finleyl/Documents/kimi/tasks/2026-11-10/20-02-04-6933ea61/ml-labs-1-5


## Проверка данных

Смотрю размеры выборок, пропуски и целевые переменные. Эти разбиения используются во всех работах.

In [2]:
# Загружаю данные и проверяю размеры выборок.
experiments = {task: TaskExperiment(5, task) for task in ("classification", "regression")}
for task, experiment in experiments.items():
    data = experiment.data
    print(f"{task}: {data.name}")
    print(data.practical_task)
    print(f"train={data.X_train.shape}, test={data.X_test.shape}, пропуски={int(data.X.isna().sum().sum())}")
    if task == "classification":
        print(f"Доля класса 1: train={data.y_train.mean():.3f}, test={data.y_test.mean():.3f}")
    else:
        print(data.y.describe().to_string())
    print()

classification: Haberman's Survival (UCI)
По возрасту пациента, году операции и числу пораженных лимфоузлов оценить риск смерти в течение пяти лет после операции.
train=(244, 3), test=(62, 3), пропуски=0
Доля класса 1: train=0.266, test=0.258

regression: QSAR aquatic toxicity (UCI)
По молекулярным дескрипторам прогнозировать токсичность LC50 для Daphnia magna и предварительно оценивать опасность веществ для водных организмов.
train=(436, 8), test=(110, 8), пропуски=0
count    546.0000
mean       4.6584
std        1.6652
min        0.1220
25%        3.6015
50%        4.5160
75%        5.6075
max       10.0470



## Бейзлайн sklearn

Беру модель с параметрами по умолчанию. Медианный imputer оставлен как страховка, хотя пропусков в данных нет.

In [3]:
# Обучаю базовые модели sklearn.
for experiment in experiments.values():
    experiment.fit_sklearn_baseline()
baseline_results = pd.concat([experiment.results for experiment in experiments.values()], ignore_index=True)
display(baseline_results.drop(columns=["params"]))

classification | sklearn | baseline: accuracy=0.7258, f1=0.3200, roc_auc=0.7038; 0.03 s
regression | sklearn | baseline: rmse=1.2244, mae=0.8578, r2=0.4717; 0.04 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,5,Градиентный бустинг,classification,sklearn,baseline,0.7258,0.32,0.7038,NaN,NaN,NaN,0.0256
1,5,Градиентный бустинг,regression,sklearn,baseline,NaN,NaN,NaN,1.2244,0.8578,0.4717,0.0379


## Подбор параметров

Подбираю скорость обучения, число деревьев и глубину. Маленький шаг обычно требует большего числа деревьев, поэтому эти параметры рассматриваются вместе.

Использую пятикратную CV. Для классификации выбираю по ROC-AUC, для регрессии по RMSE.

In [4]:
# Подбираю параметры только на обучающей выборке.
for experiment in experiments.values():
    experiment.fit_sklearn_improved()
    print(f"{{experiment.task}}: {{experiment.search.best_params_}}")
    display(experiment.cv_table().head())

classification | sklearn | improved: accuracy=0.7419, f1=0.2727, roc_auc=0.7711; 1.67 s
{experiment.task}: {experiment.search.best_params_}


,params,cv_roc_auc,std_roc_auc,mean_fit_time
0,"{'model__learning_rate': 0.03, 'model__max_dep...",0.6512,0.1155,0.0687
1,"{'model__learning_rate': 0.03, 'model__max_dep...",0.6482,0.1160,0.0434
2,"{'model__learning_rate': 0.1, 'model__max_dept...",0.6469,0.1152,0.0329
3,"{'model__learning_rate': 0.1, 'model__max_dept...",0.6390,0.1060,0.0535
4,"{'model__learning_rate': 0.03, 'model__max_dep...",0.6263,0.1340,0.0423


regression | sklearn | improved: rmse=1.2244, mae=0.8578, r2=0.4717; 0.27 s
{experiment.task}: {experiment.search.best_params_}


,params,cv_rmse,std_rmse,mean_fit_time
0,"{'model__learning_rate': 0.1, 'model__max_dept...",1.1283,0.1042,0.0434
1,"{'model__learning_rate': 0.03, 'model__max_dep...",1.1361,0.1108,0.1076
2,"{'model__learning_rate': 0.1, 'model__max_dept...",1.1383,0.1404,0.0758
3,"{'model__learning_rate': 0.1, 'model__max_dept...",1.1448,0.1095,0.0643
4,"{'model__learning_rate': 0.1, 'model__max_dept...",1.1450,0.1278,0.0362


## Что дала настройка

Сравниваю метрики до и после. Для ROC-AUC, F1, accuracy и R² рост означает улучшение, для RMSE и MAE наоборот.

In [5]:
# Считаю разницу метрик между базовой и настроенной моделями.
metric_names = {"classification": ["roc_auc", "f1", "accuracy"], "regression": ["rmse", "mae", "r2"]}
for task, experiment in experiments.items():
    rows = experiment.results.set_index("stage")
    metrics = metric_names[task]
    comparison = rows.loc[["baseline", "improved"], metrics].T
    comparison["change"] = comparison["improved"] - comparison["baseline"]
    print(task)
    display(comparison)

classification


stage,baseline,improved,change
roc_auc,0.7038,0.7711,0.0673
f1,0.3200,0.2727,-0.0473
accuracy,0.7258,0.7419,0.0161


regression


stage,baseline,improved,change
rmse,1.2244,1.2244,0.0
mae,0.8578,0.8578,0.0
r2,0.4717,0.4717,0.0


## Своя реализация

В своей версии базовые модели являются регрессионными деревьями. Их прогнозы складываются с коэффициентом `learning_rate`.

Код лежит в `src/models/`. Сначала обучаю базовую версию на тех же данных, что и sklearn.

In [6]:
# Обучаю базовые версии собственных моделей.
for experiment in experiments.values():
    experiment.fit_custom_baseline()
custom_baseline_results = pd.concat([
    experiment.results[experiment.results["implementation"] == "from_scratch"]
    for experiment in experiments.values()
], ignore_index=True)
display(custom_baseline_results.drop(columns=["params"]))

classification | from_scratch | baseline: accuracy=0.7419, f1=0.3333, roc_auc=0.7908; 0.05 s
regression | from_scratch | baseline: rmse=1.2225, mae=0.8529, r2=0.4734; 0.12 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,5,Градиентный бустинг,classification,from_scratch,baseline,0.7419,0.3333,0.7908,NaN,NaN,NaN,0.0515
1,5,Градиентный бустинг,regression,from_scratch,baseline,NaN,NaN,NaN,1.2225,0.8529,0.4734,0.1213


## Своя модель после настройки

Переношу препроцессинг и параметры, выбранные на CV. Ограничение по числу деревьев для своего леса указано выше.

In [7]:
# Применяю найденные параметры к своим моделям.
for experiment in experiments.values():
    experiment.fit_custom_improved()
all_lab_results = pd.concat([experiment.results for experiment in experiments.values()], ignore_index=True)
display(all_lab_results.drop(columns=["params"]))

classification | from_scratch | improved: accuracy=0.7581, f1=0.2105, roc_auc=0.7948; 0.06 s


regression | from_scratch | improved: rmse=1.2225, mae=0.8529, r2=0.4734; 0.12 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,5,Градиентный бустинг,classification,sklearn,baseline,0.7258,0.3200,0.7038,NaN,NaN,NaN,0.0256
1,5,Градиентный бустинг,classification,sklearn,improved,0.7419,0.2727,0.7711,NaN,NaN,NaN,1.6678
2,5,Градиентный бустинг,classification,from_scratch,baseline,0.7419,0.3333,0.7908,NaN,NaN,NaN,0.0515
3,5,Градиентный бустинг,classification,from_scratch,improved,0.7581,0.2105,0.7948,NaN,NaN,NaN,0.0578
4,5,Градиентный бустинг,regression,sklearn,baseline,NaN,NaN,NaN,1.2244,0.8578,0.4717,0.0379
5,5,Градиентный бустинг,regression,sklearn,improved,NaN,NaN,NaN,1.2244,0.8578,0.4717,0.2746
6,5,Градиентный бустинг,regression,from_scratch,baseline,NaN,NaN,NaN,1.2225,0.8529,0.4734,0.1213
7,5,Градиентный бустинг,regression,from_scratch,improved,NaN,NaN,NaN,1.2225,0.8529,0.4734,0.1190


## Результаты

Сохраняю все четыре варианта для каждой задачи. Вывод зависит от полученных чисел: если настройка не помогла, это будет видно.

In [8]:
# Сохраняю таблицу и вывожу изменение основной метрики.
output_path = ROOT / "results" / "lab05_results.csv"
output_path.parent.mkdir(exist_ok=True)
all_lab_results.to_csv(output_path, index=False)

def verdict(change, higher_is_better):
    if np.isclose(change, 0):
        return "не изменился"
    improved = change > 0 if higher_is_better else change < 0
    return "улучшился" if improved else "ухудшился"

for task, experiment in experiments.items():
    frame = experiment.results.set_index(["implementation", "stage"])
    metrics = metric_names[task]
    primary = "roc_auc" if task == "classification" else "rmse"
    higher_is_better = primary != "rmse"
    sklearn_base = frame.loc[("sklearn", "baseline"), primary]
    sklearn_best = frame.loc[("sklearn", "improved"), primary]
    custom_base = frame.loc[("from_scratch", "baseline"), primary]
    custom_best = frame.loc[("from_scratch", "improved"), primary]
    print(f"{task}, sklearn: {primary} {sklearn_base:.4f} → {sklearn_best:.4f}, "
          f"{verdict(sklearn_best - sklearn_base, higher_is_better)}.")
    print(f"{task}, своя версия: {primary} {custom_base:.4f} → {custom_best:.4f}, "
          f"{verdict(custom_best - custom_base, higher_is_better)}.")
    print(f"Параметры sklearn: {experiment.search.best_params_}")
print(f"Файл: {output_path.relative_to(ROOT)}")

classification, sklearn: roc_auc 0.7038 → 0.7711, улучшился.
classification, своя версия: roc_auc 0.7908 → 0.7948, улучшился.
Параметры sklearn: {'model__learning_rate': 0.03, 'model__max_depth': 2, 'model__n_estimators': 200}
regression, sklearn: rmse 1.2244 → 1.2244, не изменился.
regression, своя версия: rmse 1.2225 → 1.2225, не изменился.
Параметры sklearn: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100}
Файл: results/lab05_results.csv
